<a href="https://colab.research.google.com/github/SolKacil/matematicas-para-ia/blob/main/python/01-algebra-lineal/03_determinante_norma_producto_punto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 03 &middot; Determinante, norma, producto punto y ortogonalidad

**Módulo 1 — Álgebra lineal y geometría diferencial**

Este notebook acompaña al documento `Matematicas_para_IA_03.pdf`, basado en Deisenroth, Faisal y
Ong (2020), *Mathematics for Machine Learning*, secciones 3.1–3.4 y 4.1.

Los notebooks anteriores trataron operaciones y estructuras. Éste introduce **medida**: longitudes,
ángulos, áreas y volúmenes. Con ello, un espacio vectorial deja de ser sólo un conjunto donde se
puede sumar y escalar, y pasa a ser un espacio donde tiene sentido preguntar qué tan largo es un
vector, qué tan parecidos son dos de ellos y cuánto distorsiona el espacio una transformación.

Las cuatro nociones que se definen están estrechamente relacionadas, y conviene ver desde el
principio el mapa: el **producto punto** es la operación primitiva; la **norma** es el producto
punto de un vector consigo mismo; la **ortogonalidad** es el caso en que el producto punto se anula;
las **matrices ortogonales** son aquellas cuyas columnas son ortonormales; y el **determinante**
mide el factor de escalamiento de volumen, que para una matriz ortogonal vale $\pm 1$ precisamente
porque una rotación no altera volúmenes.

Cada una tiene un uso directo en aprendizaje automático: la similitud del coseno y los mecanismos de
atención son productos punto normalizados; la regularización $L_2$ penaliza una norma; la
inicialización ortogonal estabiliza el entrenamiento de redes profundas; y el determinante del
jacobiano hace posible evaluar densidades en modelos generativos.

## Al terminar será posible

- Calcular **determinantes** de $2\times2$ y $3\times3$ a mano y con NumPy, e interpretarlos como
  área y volumen con signo.
- Relacionar $\det(A) = 0$ con la singularidad de $A$ y con la dependencia lineal de sus columnas.
- Calcular **normas** y **productos punto**, y obtener el **ángulo** entre dos vectores.
- Decidir si dos vectores son **ortogonales**, **normalizar** un vector y reconocer un conjunto
  **ortonormal**.
- Verificar que una matriz es **ortogonal**, invertirla por transposición y comprobar que preserva
  normas y ángulos.

## Qué se da por sabido

Los notebooks [00 · Propiedades de matrices](00_propiedades_matrices.ipynb) —en particular la
transpuesta— y [02 · Sistemas lineales](02_sistemas_lineales_y_espacio_nulo.ipynb), del que se
reutiliza la relación entre singularidad, dependencia lineal y rango.

## Cómo usar este notebook

1. Con el botón **Open in Colab** no se requiere instalación alguna.
2. Las celdas se ejecutan en orden con `Shift + Enter`.
3. Cada fórmula se implementa primero **a partir de su definición** y después se contrasta con la
   rutina de NumPy: escribirla a mano una vez es lo que fija el contenido de la fórmula.
4. La sección final, **Tu turno**, contiene los ejercicios propuestos del documento.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (5.5, 5.5)

print("numpy:", np.__version__)

---

## 1. El determinante

### 1.1 Qué es y cómo se calcula

El determinante es un número asociado a una matriz **cuadrada** que resume dos propiedades:
indica si la matriz tiene inversa —no la tiene exactamente cuando el determinante vale cero— y
cuantifica el factor por el que la transformación asociada escala áreas o volúmenes. Se denota
$\det(A)$ o $|A|$.

Para una matriz $2 \times 2$:

$$A = \begin{bmatrix} a & b \\ c & d \end{bmatrix} \;\Longrightarrow\; \det(A) = ad - bc$$

Con $A = \begin{bmatrix} 3 & 1 \\ 2 & 4 \end{bmatrix}$ se obtiene $\det(A) = (3)(4) - (1)(2) = 10$.

In [ ]:
def det2(A):
    """Determinante de una matriz 2x2 a partir de su definicion: ad - bc."""
    (a, b), (c, d) = A
    return a * d - b * c


A = np.array([[3.0, 1.0],
              [2.0, 4.0]])

print("A =\n", A)
print("\ndet a mano  :", det2(A))
print("np.linalg.det:", np.linalg.det(A))

B = np.array([[1.0, 2.0],
              [2.0, 4.0]])
print("\nB =\n", B)
print("det(B) =", det2(B), " -> B es singular")
print("rango de B:", np.linalg.matrix_rank(B), "< 2, y la columna 2 es 2 veces la columna 1")

### 1.2 Significado geométrico

Si se toman las columnas de $A$ como vectores del plano, el paralelogramo que determinan tiene área
exactamente $|\det(A)|$. Para $A = \begin{bmatrix} 3 & 1 \\ 2 & 4 \end{bmatrix}$, las columnas
$(3,2)$ y $(1,4)$ generan un paralelogramo de área $10$.

Cuando las dos columnas son paralelas —una múltiplo de la otra— el paralelogramo degenera en un
segmento, de área nula. Esto conecta el determinante con los conceptos del notebook 02:

$$\det(A) = 0 \iff A \text{ es singular} \iff \text{las columnas de } A \text{ son linealmente dependientes}.$$

El **signo** del determinante también tiene contenido geométrico: indica si la transformación
preserva o invierte la orientación del plano. Por eso el área es el valor absoluto.

In [ ]:
def dibujar_paralelogramo(ax, M, titulo):
    c1, c2 = M[:, 0], M[:, 1]
    esquinas = np.array([[0, 0], c1, c1 + c2, c2, [0, 0]])
    ax.fill(esquinas[:, 0], esquinas[:, 1], alpha=0.25, color="tab:blue")
    ax.plot(esquinas[:, 0], esquinas[:, 1], color="tab:blue", lw=1)
    for v, nombre, color in [(c1, "c1", "tab:red"), (c2, "c2", "tab:green")]:
        ax.quiver(0, 0, v[0], v[1], angles="xy", scale_units="xy", scale=1, color=color, width=0.012)
        ax.annotate(f"{nombre} = ({v[0]:g}, {v[1]:g})", v, textcoords="offset points",
                    xytext=(6, 4), color=color, fontsize=9)
    ax.set_xlim(-1, 6); ax.set_ylim(-1, 6); ax.set_aspect("equal")
    ax.axhline(0, color="gray", lw=0.8); ax.axvline(0, color="gray", lw=0.8)
    ax.grid(alpha=0.3); ax.set_title(titulo, fontsize=10)


fig, ejes = plt.subplots(1, 2, figsize=(11, 5.5))
dibujar_paralelogramo(ejes[0], A, f"det(A) = {det2(A):g}  ->  area = {abs(det2(A)):g}")
dibujar_paralelogramo(ejes[1], B, f"det(B) = {det2(B):g}  ->  area = 0 (degenerado)")
plt.show()

In [ ]:
# El signo indica la orientacion: intercambiar dos columnas la invierte.
A_intercambiada = A[:, [1, 0]]
print("det(A)               =", det2(A))
print("det(A con columnas intercambiadas) =", det2(A_intercambiada))
print("misma area, orientacion opuesta")

# El determinante escala con el area: al duplicar una columna, el area se duplica.
A_escalada = A.copy()
A_escalada[:, 0] *= 2
print("\nal duplicar la primera columna, det pasa de", det2(A), "a", det2(A_escalada))

### 1.3 Matrices $3 \times 3$ y propiedades

Para una matriz $3\times3$, la regla de Sarrus da

$$\det(A) = a_{11}a_{22}a_{33} + a_{12}a_{23}a_{31} + a_{13}a_{21}a_{32}
- a_{13}a_{22}a_{31} - a_{11}a_{23}a_{32} - a_{12}a_{21}a_{33}.$$

La interpretación es la misma un grado más arriba: el determinante mide el **volumen** del
paralelepípedo generado por las tres columnas.

**Propiedades.**

- $\det(A) = 0 \iff A$ no tiene inversa.
- $\det(AB) = \det(A)\det(B)$.
- $\det(A^\top) = \det(A)$.
- $\det(A)$ es el **producto de todos los autovalores** de $A$ — la conexión que se retoma en el
  notebook 04.

In [ ]:
def det3_sarrus(A):
    """Determinante 3x3 por la regla de Sarrus, escrita termino a termino."""
    a = A
    return (a[0, 0] * a[1, 1] * a[2, 2] + a[0, 1] * a[1, 2] * a[2, 0]
            + a[0, 2] * a[1, 0] * a[2, 1] - a[0, 2] * a[1, 1] * a[2, 0]
            - a[0, 0] * a[1, 2] * a[2, 1] - a[0, 1] * a[1, 0] * a[2, 2])


M = np.array([[2.0, 0.0, 1.0],
              [1.0, 3.0, 2.0],
              [0.0, 1.0, 1.0]])

print("Sarrus      :", det3_sarrus(M))
print("np.linalg.det:", np.linalg.det(M))
print("volumen del paralelepipedo:", abs(det3_sarrus(M)))

In [ ]:
rng = np.random.default_rng(0)
P = rng.normal(size=(3, 3))
Q = rng.normal(size=(3, 3))

print(f"det(P@Q)        = {np.linalg.det(P @ Q): .6f}")
print(f"det(P) * det(Q) = {np.linalg.det(P) * np.linalg.det(Q): .6f}")
print(f"det(P.T)        = {np.linalg.det(P.T): .6f}   det(P) = {np.linalg.det(P): .6f}")

# El determinante es el producto de los autovalores (notebook 04).
autovalores = np.linalg.eigvals(P)
print(f"\nautovalores: {np.round(autovalores, 4)}")
print(f"producto de los autovalores = {np.prod(autovalores).real: .6f}")

> **Advertencia numérica.** Aunque $\det(A) = 0$ caracteriza la singularidad en aritmética exacta,
> el determinante es un **mal indicador numérico** de que una matriz esté cerca de ser singular,
> porque escala con el tamaño de las entradas y con la dimensión: la matriz $0.1 \cdot I_{20}$ tiene
> determinante $10^{-20}$ y es perfectamente invertible. El diagnóstico fiable es el **número de
> condición** o el rango numérico calculado por SVD, como se vio en el notebook 02.

In [ ]:
pequena = 0.1 * np.eye(20)
print(f"det(0.1 * I_20)  = {np.linalg.det(pequena):.3e}   <- diminuto")
print(f"cond(0.1 * I_20) = {np.linalg.cond(pequena):.3f}   <- perfectamente invertible")
print("rango:", np.linalg.matrix_rank(pequena), "de 20")

casi_singular = np.array([[1.0, 2.0],
                          [2.0, 4.0 + 1e-12]])
print(f"\ndet(casi singular)  = {np.linalg.det(casi_singular):.3e}")
print(f"cond(casi singular) = {np.linalg.cond(casi_singular):.3e}   <- la senal de alarma")

---

## 2. Norma euclidiana

La **norma** de un vector es su longitud. Para $\mathbf{x} = (x_1, \dots, x_n)$:

$$\|\mathbf{x}\| = \sqrt{x_1^2 + x_2^2 + \dots + x_n^2}$$

En dos dimensiones es el teorema de Pitágoras: la hipotenusa del triángulo rectángulo de catetos
$x_1$ y $x_2$. Para $\mathbf{x} = (3,4)$ resulta $\|\mathbf{x}\| = \sqrt{9 + 16} = 5$.

La norma euclidiana es una entre varias. La familia $L_p$ incluye también
$\|\mathbf{x}\|_1 = \sum_i |x_i|$ y $\|\mathbf{x}\|_\infty = \max_i |x_i|$, que aparecen en
regularización y en análisis de robustez; NumPy las calcula con el argumento `ord`.

In [ ]:
def norma(x):
    """Norma euclidiana a partir de la definicion."""
    return np.sqrt(sum(xi ** 2 for xi in x))


x = np.array([3.0, 4.0])
print("x =", x)
print("norma a mano   :", norma(x))
print("np.linalg.norm :", np.linalg.norm(x))

print("\notras normas del mismo vector:")
print("  L1  (suma de valores absolutos):", np.linalg.norm(x, ord=1))
print("  L2  (euclidiana)               :", np.linalg.norm(x, ord=2))
print("  Linf (maximo valor absoluto)   :", np.linalg.norm(x, ord=np.inf))

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.quiver(0, 0, x[0], x[1], angles="xy", scale_units="xy", scale=1, color="tab:blue", width=0.013)
ax.plot([0, x[0]], [0, 0], "--", color="tab:gray")
ax.plot([x[0], x[0]], [0, x[1]], "--", color="tab:gray")
ax.annotate("3", (1.5, -0.35), color="tab:gray")
ax.annotate("4", (3.1, 2.0), color="tab:gray")
ax.annotate(r"$\|x\| = 5$", (1.2, 2.4), color="tab:blue", fontsize=12)
ax.annotate("(3, 4)", x, textcoords="offset points", xytext=(6, 4), color="tab:blue")
ax.set_xlim(-1, 5.5); ax.set_ylim(-1, 5.5); ax.set_aspect("equal")
ax.axhline(0, color="gray", lw=0.8); ax.axvline(0, color="gray", lw=0.8)
ax.grid(alpha=0.3); ax.set_title("La norma es la hipotenusa")
plt.show()

---

## 3. Producto punto

El **producto punto** combina dos vectores del mismo tamaño y devuelve un **número**, no un vector.
Se denota $\mathbf{x} \cdot \mathbf{y}$ o $\mathbf{x}^\top\mathbf{y}$:

$$\mathbf{x} \cdot \mathbf{y} = x_1y_1 + x_2y_2 + \dots + x_ny_n$$

De la definición se sigue inmediatamente que $\mathbf{x} \cdot \mathbf{x} = \|\mathbf{x}\|^2$: la
norma es un caso particular del producto punto.

### 3.1 Interpretación geométrica

$$\mathbf{x} \cdot \mathbf{y} = \|\mathbf{x}\|\,\|\mathbf{y}\|\cos(\theta)
\qquad\Longrightarrow\qquad
\cos(\theta) = \frac{\mathbf{x} \cdot \mathbf{y}}{\|\mathbf{x}\|\,\|\mathbf{y}\|}$$

donde $\theta$ es el ángulo entre ambos vectores. El producto punto mide, por tanto, **qué tan
alineados están**: es máximo cuando apuntan en la misma dirección, nulo cuando son perpendiculares y
negativo cuando forman un ángulo obtuso.

Con $\mathbf{x} = (1,0)$ y $\mathbf{y} = (1,1)$ se obtiene
$\cos(\theta) = 1/\sqrt{2} \approx 0.707$, es decir $\theta = 45°$.

In [ ]:
def producto_punto(x, y):
    """Producto punto a partir de la definicion: suma de productos componente a componente."""
    return sum(xi * yi for xi, yi in zip(x, y))


def angulo(x, y, en_grados=True):
    """Angulo entre dos vectores, despejado de la formula del producto punto."""
    coseno = producto_punto(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))
    coseno = np.clip(coseno, -1.0, 1.0)         # protege contra el redondeo
    theta = np.arccos(coseno)
    return np.degrees(theta) if en_grados else theta


u = np.array([1.0, 0.0])
v = np.array([1.0, 1.0])

print("u . v a mano :", producto_punto(u, v))
print("np.dot       :", np.dot(u, v), "   u @ v:", u @ v)
print(f"\ncos(theta) = {producto_punto(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)):.4f}")
print(f"theta      = {angulo(u, v):.1f} grados")

print("\nx . x == ||x||^2:", producto_punto(x, x), "==", np.linalg.norm(x) ** 2)

In [ ]:
# El producto punto ordena los pares por alineacion.
referencia = np.array([1.0, 0.0])
casos = {
    "misma direccion": np.array([3.0, 0.0]),
    "45 grados": np.array([1.0, 1.0]),
    "perpendicular": np.array([0.0, 2.0]),
    "obtuso": np.array([-1.0, 1.0]),
    "opuesto": np.array([-2.0, 0.0]),
}

for nombre, w in casos.items():
    print(f"{nombre:>16}: u . w = {producto_punto(referencia, w):>6.2f}   "
          f"cos = {producto_punto(referencia, w) / (np.linalg.norm(referencia) * np.linalg.norm(w)):>6.3f}   "
          f"angulo = {angulo(referencia, w):>5.1f} grados")

### 3.2 El caso en que el producto punto se anula

Si $\mathbf{x} \cdot \mathbf{y} = 0$ entonces $\cos(\theta) = 0$, es decir $\theta = 90°$: los
vectores son perpendiculares. El caso es lo bastante importante como para tener nombre propio,
**ortogonalidad**, y lo bastante útil como para que la comprobación se haga siempre por esta vía y
no midiendo ángulos: con $\mathbf{x} = (2,3)$ y $\mathbf{y} = (3,-2)$ basta observar que
$(2)(3) + (3)(-2) = 0$.

In [ ]:
p = np.array([2.0, 3.0])
q = np.array([3.0, -2.0])

print("p . q =", producto_punto(p, q), " -> ortogonales")
print(f"angulo: {angulo(p, q):.1f} grados")

# La comprobacion se generaliza a cualquier dimension, donde no hay nada que dibujar.
rng = np.random.default_rng(1)
a = rng.normal(size=50)
b = rng.normal(size=50)
b_ortogonal = b - (a @ b) / (a @ a) * a          # se le quita a b su componente sobre a

print(f"\nen R^50:  a . b = {a @ b:.4f}   angulo = {angulo(a, b):.1f} grados")
print(f"          a . b_ortogonal = {a @ b_ortogonal:.2e}   angulo = {angulo(a, b_ortogonal):.1f} grados")

### Lectura en aprendizaje automático

La **similitud del coseno** entre dos representaciones vectoriales es literalmente la fórmula
anterior:

$$\text{sim}(\mathbf{x}, \mathbf{y}) = \frac{\mathbf{x} \cdot \mathbf{y}}{\|\mathbf{x}\|\,\|\mathbf{y}\|}.$$

Es la métrica habitual para comparar *embeddings* en búsqueda semántica y sistemas de recomendación.
Se prefiere al producto punto sin normalizar porque elimina el efecto de la magnitud —que en un
*embedding* suele reflejar la frecuencia del término más que su significado— y deja únicamente la
dirección.

El mismo cálculo está en el núcleo del **mecanismo de atención** de los Transformers. Su primer paso
consiste en calcular productos punto entre vectores de consulta y de clave, escalarlos y
normalizarlos con `softmax`:

$$\text{Atención}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V .$$

La matriz $QK^\top$ contiene todos los productos punto entre consultas y claves; el factor
$1/\sqrt{d_k}$ compensa que la varianza del producto punto crezca con la dimensión.

In [ ]:
def similitud_coseno(X, Y):
    """Matriz de similitudes coseno entre las filas de X y las de Y."""
    Xn = X / np.linalg.norm(X, axis=1, keepdims=True)
    Yn = Y / np.linalg.norm(Y, axis=1, keepdims=True)
    return Xn @ Yn.T


# Cuatro representaciones: las dos primeras apuntan casi en la misma direccion.
emb = np.array([[1.0, 0.2, 0.1],
                [0.9, 0.3, 0.15],
                [0.1, 1.0, 0.0],
                [-1.0, -0.2, -0.1]])
etiquetas = ["doc A", "doc B", "doc C", "doc D"]

S = similitud_coseno(emb, emb)
print("matriz de similitudes coseno:")
print("        " + "  ".join(f"{e:>6}" for e in etiquetas))
for i, e in enumerate(etiquetas):
    print(f"{e:>6}  " + "  ".join(f"{s:>6.3f}" for s in S[i]))

print("\nA y B son casi iguales; D es el opuesto de A; C es casi ortogonal a A.")

In [ ]:
def softmax(Z, eje=-1):
    Z = Z - Z.max(axis=eje, keepdims=True)       # estabilidad numerica
    E = np.exp(Z)
    return E / E.sum(axis=eje, keepdims=True)


def atencion(Q, K, V):
    """Atencion por producto punto escalado: softmax(Q K^T / sqrt(d_k)) V."""
    d_k = Q.shape[-1]
    puntuaciones = Q @ K.T / np.sqrt(d_k)        # todos los productos punto consulta-clave
    pesos = softmax(puntuaciones)
    return pesos @ V, pesos


rng = np.random.default_rng(3)
Q = rng.normal(size=(2, 4))          # 2 consultas de dimension 4
K = rng.normal(size=(5, 4))          # 5 claves
V = rng.normal(size=(5, 3))          # 5 valores de dimension 3

salida, pesos = atencion(Q, K, V)
print("puntuaciones Q @ K.T / sqrt(d_k):\n", Q @ K.T / np.sqrt(4))
print("\npesos de atencion (cada fila suma 1):\n", pesos)
print("\nsuma por fila:", pesos.sum(axis=1))
print("\nsalida:", salida.shape, "-> una combinacion de los valores por cada consulta")

# El factor 1/sqrt(d_k) evita que las puntuaciones crezcan con la dimension.
for d in [4, 64, 1024]:
    q, k = rng.normal(size=(1000, d)), rng.normal(size=(1000, d))
    bruto = (q * k).sum(axis=1)
    print(f"\nd_k = {d:>5}: desviacion del producto punto = {bruto.std():>7.2f}   "
          f"escalado = {(bruto / np.sqrt(d)).std():.2f}")

### Lectura en aprendizaje automático: la norma

La **regularización $L_2$** —también llamada *weight decay*— añade a la función de pérdida un
término proporcional al cuadrado de la norma de los parámetros:

$$L_{\text{total}}(\boldsymbol{\theta}) = L(\boldsymbol{\theta}) + \lambda\|\boldsymbol{\theta}\|^2 .$$

El término es exactamente $\boldsymbol{\theta} \cdot \boldsymbol{\theta}$, y su efecto se ve mejor en
el gradiente: como $\nabla_{\boldsymbol{\theta}} \lambda\|\boldsymbol{\theta}\|^2 =
2\lambda\boldsymbol{\theta}$, cada paso de descenso contrae los pesos hacia el origen en una
proporción fija, de ahí el nombre. La consecuencia es que, entre las soluciones que ajustan los
datos por igual, se prefiere la de menor norma, lo cual enlaza con la solución de norma mínima del
notebook 02.

Sustituir $L_2$ por $L_1$ cambia cualitativamente el resultado: el gradiente de
$\|\boldsymbol{\theta}\|_1$ tiene magnitud constante, de modo que empuja las componentes hasta
anularlas exactamente y produce soluciones **dispersas**. La elección de la norma no es un detalle
de implementación, sino una decisión sobre qué clase de solución se busca.

In [ ]:
# Efecto de la penalizacion L2 sobre un ajuste por minimos cuadrados.
rng = np.random.default_rng(5)
X = rng.normal(size=(15, 8))
y = X @ np.array([2.0, -1.0, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0]) + 0.1 * rng.normal(size=15)

for lam in [0.0, 0.1, 1.0, 10.0]:
    # Solucion de ridge: (X^T X + lam I)^-1 X^T y, resuelta sin invertir explicitamente.
    theta = np.linalg.solve(X.T @ X + lam * np.eye(8), X.T @ y)
    error = np.linalg.norm(X @ theta - y) ** 2
    print(f"lambda = {lam:>5}: ||theta|| = {np.linalg.norm(theta):>6.3f}   "
          f"error de ajuste = {error:>8.4f}")

print("\nal crecer lambda, la norma baja y el error de ajuste sube: es el compromiso que impone la regularizacion")

---

## 4. Ortogonalidad y matrices ortogonales

### 4.1 Vectores ortogonales, unitarios y ortonormales

- Dos vectores son **ortogonales** si $\mathbf{x} \cdot \mathbf{y} = 0$.
- Un vector es **unitario** si $\|\mathbf{x}\| = 1$. Cualquier vector no nulo se **normaliza**
  dividiéndolo entre su propia norma, lo que conserva su dirección:
  $\hat{\mathbf{x}} = \mathbf{x}/\|\mathbf{x}\|$.
- Un conjunto es **ortonormal** si todos sus vectores son unitarios y ortogonales entre sí.

In [ ]:
def normalizar(x):
    """Devuelve el vector unitario con la misma direccion."""
    return np.asarray(x, dtype=float) / np.linalg.norm(x)


x = np.array([3.0, 4.0])
x_hat = normalizar(x)
print("x =", x, "  ||x|| =", np.linalg.norm(x))
print("x normalizado =", x_hat, "  ||x_hat|| =", np.linalg.norm(x_hat))
print("misma direccion:", np.allclose(angulo(x, x_hat), 0.0))

# Un conjunto ortonormal: la matriz de productos punto es la identidad.
e1 = np.array([1.0, 0.0])
e2 = np.array([0.0, 1.0])
base = np.column_stack([e1, e2])
print("\nmatriz de productos punto de {e1, e2}:\n", base.T @ base)

### 4.2 Matrices ortogonales

Una matriz cuadrada $A$ es **ortogonal** si sus columnas forman un conjunto ortonormal, condición
que se resume en

$$A^\top A = I .$$

La igualdad recoge las dos exigencias a la vez: las entradas de la diagonal de $A^\top A$ son los
productos $\mathbf{c}_i \cdot \mathbf{c}_i = \|\mathbf{c}_i\|^2$, que deben valer $1$, y las de
fuera de la diagonal son $\mathbf{c}_i \cdot \mathbf{c}_j$, que deben anularse.

La matriz de rotación de $90°$ es el ejemplo mínimo:

$$R = \begin{bmatrix} 0 & -1 \\ 1 & 0 \end{bmatrix}$$

Sus columnas $(0,1)$ y $(-1,0)$ son unitarias y su producto punto es cero.

### 4.3 La consecuencia más útil

De $A^\top A = I$ se sigue directamente, por definición de inversa, que

$$A^{-1} = A^\top .$$

Invertir una matriz ortogonal no requiere eliminación de Gauss-Jordan ni ningún otro cálculo: basta
transponer. Además, las transformaciones ortogonales **preservan normas, productos punto, ángulos y
distancias**, y su determinante vale $\pm 1$: no deforman el espacio, sólo lo rotan o lo reflejan.

In [ ]:
R = np.array([[0.0, -1.0],
              [1.0, 0.0]])

print("R.T @ R =\n", R.T @ R)
print("\nes ortogonal:", np.allclose(R.T @ R, np.eye(2)))
print("inversa == transpuesta:", np.allclose(np.linalg.inv(R), R.T))
print("\nR^-1 = R.T =\n", R.T)
print("\ndet(R) =", np.linalg.det(R), " -> |det| = 1, no cambia areas")

In [ ]:
# Las transformaciones ortogonales preservan normas, productos punto y angulos.
rng = np.random.default_rng(7)
u = rng.normal(size=2)
v = rng.normal(size=2)

print(f"||u||     = {np.linalg.norm(u):.6f}      ||R u||     = {np.linalg.norm(R @ u):.6f}")
print(f"u . v     = {u @ v:.6f}      (Ru).(Rv)   = {(R @ u) @ (R @ v):.6f}")
print(f"angulo    = {angulo(u, v):.4f}     tras rotar  = {angulo(R @ u, R @ v):.4f}")
print(f"distancia = {np.linalg.norm(u - v):.6f}      tras rotar  = {np.linalg.norm(R @ u - R @ v):.6f}")

# En cualquier dimension se construye una matriz ortogonal con la factorizacion QR.
Z = rng.normal(size=(5, 5))
Q, _ = np.linalg.qr(Z)
print("\nQ de 5x5 por QR:  Q.T @ Q == I:", np.allclose(Q.T @ Q, np.eye(5)))
print("det(Q) =", round(float(np.linalg.det(Q)), 10))
print("norma preservada:", np.allclose(np.linalg.norm(Q @ rng.normal(size=5)),
                                       np.linalg.norm(rng.normal(size=5))) or "ver celda siguiente")

### Lectura en aprendizaje automático

Que las matrices ortogonales preserven la norma tiene una consecuencia directa sobre el
entrenamiento de redes profundas. Al propagar una señal por $L$ capas lineales, la norma se
multiplica en cada una por un factor que depende de los valores singulares de la matriz de pesos;
si ese factor es sistemáticamente mayor que uno, la norma crece de forma exponencial con la
profundidad, y si es menor, decae. Ése es el fenómeno de los **gradientes que explotan o se
desvanecen**.

Una matriz ortogonal tiene todos sus valores singulares iguales a $1$, de modo que la norma se
conserva exactamente capa tras capa. De ahí la **inicialización ortogonal**, habitual en redes
recurrentes, donde la misma matriz se aplica una vez por paso temporal y el efecto acumulativo es
más severo. La celda siguiente contrasta ambos comportamientos.

In [ ]:
rng = np.random.default_rng(11)
n, capas = 64, 60
senal = rng.normal(size=n)

W_gauss = rng.normal(size=(n, n)) * 0.15            # escala arbitraria
W_ortogonal = np.linalg.qr(rng.normal(size=(n, n)))[0]

normas_g, normas_o = [np.linalg.norm(senal)], [np.linalg.norm(senal)]
xg = xo = senal
for _ in range(capas):
    xg, xo = W_gauss @ xg, W_ortogonal @ xo
    normas_g.append(np.linalg.norm(xg))
    normas_o.append(np.linalg.norm(xo))

print(f"norma inicial: {normas_g[0]:.4f}")
print(f"tras {capas} capas gaussianas : {normas_g[-1]:.3e}")
print(f"tras {capas} capas ortogonales: {normas_o[-1]:.4f}   <- inalterada")
print("\nvalores singulares de W ortogonal:", np.round(np.linalg.svd(W_ortogonal, compute_uv=False)[:5], 6), "...")

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.semilogy(normas_g, "o-", ms=3, color="tab:red", label="pesos gaussianos")
ax.semilogy(normas_o, "o-", ms=3, color="tab:blue", label="pesos ortogonales")
ax.set_xlabel("capa"); ax.set_ylabel("norma de la senal (escala logaritmica)")
ax.set_title("Las transformaciones ortogonales preservan la norma")
ax.grid(alpha=0.3); ax.legend()
plt.show()

---

## Tu turno

Los ejercicios son los propuestos en el documento `Matematicas_para_IA_03.pdf`. Sus respuestas están
en el PDF; el objetivo aquí es **calcularlas a mano primero y verificarlas después con código**,
usando las funciones `det2`, `norma`, `producto_punto`, `angulo` y `normalizar` definidas arriba.

**Determinante**

**1.** Calcular $\det(A)$ para $A = \begin{bmatrix} 5 & 2 \\ 3 & 4 \end{bmatrix}$ e indicar si $A$
tiene inversa. Verificar con `np.linalg.inv` que la inversa existe y que $AA^{-1} = I$.

**2.** ¿Es $C = \begin{bmatrix} 2 & 6 \\ 1 & 3 \end{bmatrix}$ invertible? Calcular el determinante,
el rango, y comprobar la relación entre sus columnas.

**3.** Dibujar el paralelogramo generado por las columnas de $A$ del ejercicio 1 con la función
`dibujar_paralelogramo` y confirmar que su área coincide con $|\det(A)|$.

**Norma y producto punto**

**4.** Calcular $\|\mathbf{x}\|$ para $\mathbf{x} = (6, 8)$.

**5.** Calcular el ángulo entre $\mathbf{x} = (1, 0)$ y $\mathbf{y} = (1, \sqrt{3})$. Verificar que
`angulo` devuelve $60°$.

**6.** Comprobar con código la desigualdad de Cauchy-Schwarz,
$|\mathbf{x} \cdot \mathbf{y}| \leq \|\mathbf{x}\|\,\|\mathbf{y}\|$, sobre cien pares de vectores
aleatorios de $\mathbb{R}^{10}$, e identificar en qué caso se alcanza la igualdad.

**Ortogonalidad**

**7.** ¿Son ortogonales $\mathbf{x} = (4, -1)$ y $\mathbf{y} = (1, 4)$?

**8.** Normalizar $\mathbf{x} = (3, 4)$ y verificar que el resultado tiene norma $1$.

**9.** Verificar que la matriz de rotación general
$R(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$
es ortogonal para cualquier $\theta$, y comprobar numéricamente que
$R(\alpha)R(\beta) = R(\alpha + \beta)$.

**10.** Dados $\mathbf{a} = (1, 1, 0)$ y $\mathbf{b} = (2, 0, 1)$, construir un conjunto ortonormal
de dos vectores que genere el mismo plano, restándole a $\mathbf{b}$ su componente en la dirección
de $\mathbf{a}$ y normalizando después ambos. Verificar que la matriz de productos punto es la
identidad de $2 \times 2$.

In [ ]:
# Tu codigo aqui.
A_ej = np.array([[5.0, 2.0],
                 [3.0, 4.0]])

# 1.

---

## Resumen

| Concepto | Definición | A mano | En NumPy |
|---|---|---|---|
| Determinante $2\times2$ | $ad - bc$ | `det2` | `np.linalg.det` |
| Determinante $3\times3$ | regla de Sarrus | `det3_sarrus` | `np.linalg.det` |
| Norma | $\sqrt{\sum_i x_i^2}$ | `norma` | `np.linalg.norm` |
| Producto punto | $\sum_i x_iy_i$ | `producto_punto` | `np.dot`, `x @ y` |
| Ángulo | $\arccos\dfrac{\mathbf{x}\cdot\mathbf{y}}{\|\mathbf{x}\|\|\mathbf{y}\|}$ | `angulo` | — |
| Normalizar | $\mathbf{x}/\|\mathbf{x}\|$ | `normalizar` | — |
| Matriz ortogonal | $A^\top A = I$ | — | `np.linalg.qr` |

| Relación | Enunciado |
|---|---|
| Determinante y singularidad | $\det(A) = 0 \iff A$ singular $\iff$ columnas dependientes |
| Norma y producto punto | $\mathbf{x} \cdot \mathbf{x} = \|\mathbf{x}\|^2$ |
| Producto punto y ángulo | $\mathbf{x} \cdot \mathbf{y} = \|\mathbf{x}\|\|\mathbf{y}\|\cos\theta$ |
| Ortogonalidad | $\mathbf{x} \cdot \mathbf{y} = 0 \iff \theta = 90°$ |
| Matriz ortogonal | $A^{-1} = A^\top$, preserva normas y ángulos, $|\det| = 1$ |

Cuatro ideas para retener:

1. **El producto punto es la operación primitiva.** Norma, ángulo y ortogonalidad se derivan de él.
2. **El determinante mide volumen con signo.** Que se anule significa que la transformación aplasta
   el espacio, y por eso equivale a la singularidad.
3. **El determinante no sirve como diagnóstico numérico.** Para decidir si una matriz está cerca de
   ser singular se usa el número de condición o los valores singulares.
4. **Las matrices ortogonales son las que no deforman.** Se invierten transponiendo y conservan la
   norma, que es la razón de su uso en inicialización de redes profundas.

## Qué sigue

- La versión en script, más corta y sin explicaciones: [`03_determinante_norma_producto_punto.py`](03_determinante_norma_producto_punto.py)
- El documento de la sesión: [`Matematicas_para_IA_03.pdf`](Matematicas_para_IA_03.pdf)
- El notebook anterior: [`02 · Sistemas lineales, espacio nulo y rango`](02_sistemas_lineales_y_espacio_nulo.ipynb)
- El siguiente notebook: [`04 · Autovalores y autovectores`](04_autovalores_y_autovectores.ipynb)
- Los demás módulos, en el [README del repositorio](../../README.md)

**Referencia.** Deisenroth, M. P., Faisal, A. A. y Ong, C. S. (2020). *Mathematics for Machine
Learning*. Cambridge University Press, secciones 3.1–3.4 y 4.1.

---

*Material abierto bajo licencia MIT. ¿Encontraste un error o quieres aportar un ejercicio?
Lee [CONTRIBUTING.md](../../CONTRIBUTING.md).*